# 01 Crop Tree Crowns
Extract individual tree crowns from drone orthomosaics using crown polygons.

In [ ]:
import os
import rasterio
import geopandas as gpd
from shapely.geometry import mapping
from rasterio.mask import mask
from tqdm import tqdm

## Configuration

In [ ]:
ORTHO_FOLDER = "data/orthomosaics"
POLYGON_FOLDER = "data/crown_polygons"
OUTPUT_DIR = "outputs/cropped_crowns"

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Crop tree crowns

In [ ]:
for gj in os.listdir(POLYGON_FOLDER):

    if not gj.endswith(".geojson"):
        continue

    prefix = os.path.splitext(gj)[0]
    gdf = gpd.read_file(os.path.join(POLYGON_FOLDER, gj))

    for ortho in os.listdir(ORTHO_FOLDER):

        if not ortho.endswith(".tif"):
            continue

        ortho_path = os.path.join(ORTHO_FOLDER, ortho)

        with rasterio.open(ortho_path) as src:

            if gdf.crs != src.crs:
                gdf = gdf.to_crs(src.crs)

            for idx, row in tqdm(gdf.iterrows(), total=len(gdf)):

                crown_id = row["id"] if "id" in row else idx
                filename = f"{prefix}_{int(crown_id):03d}.tif"
                out_path = os.path.join(OUTPUT_DIR, filename)

                try:
                    out_img, out_transform = mask(src, [mapping(row.geometry)], crop=True)

                    meta = src.meta.copy()
                    meta.update({
                        "height": out_img.shape[1],
                        "width": out_img.shape[2],
                        "transform": out_transform
                    })

                    with rasterio.open(out_path, "w", **meta) as dst:
                        dst.write(out_img)

                except:
                    continue